# batchnorm-affine-params — worked example 1: Apply BatchNorm affine to a (B, C, L) 1-D tensor

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `batchnorm-affine-params`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

BatchNorm's affine step is `y = gamma * x_hat + beta`, applied **per channel**. The learnable params `gamma` and `beta` each have shape `(C,)`. To broadcast them against a tensor whose channel axis is *not* the last axis, you reshape so the `C` lines up and the other axes become size-1.

## Worked solution

We have a 1-D-conv-style tensor `x_hat` of shape `(B, C, L)` (batch, channels, length) already normalized to mean 0 / var 1 per channel, plus per-channel `gamma` and `beta` of shape `(C,)`.

**Step 1 — identify which axis is the channel axis.** Here it is axis 1 (the middle one). Broadcasting in PyTorch/NumPy aligns shapes from the *right*, so a raw `(C,)` tensor would try to line up against `L` (the last axis), which is wrong.

**Step 2 — reshape the params to `(1, C, 1)`.** A size-1 axis broadcasts freely. So `gamma.view(1, -1, 1)` makes a `(1, C, 1)` tensor that lines up the `C` against axis 1 of `x_hat` and stretches over `B` and `L`. This is the exact analogue of the `(1, C, 1, 1)` reshape used for the 4-D `(B, C, H, W)` case — one fewer spatial axis.

**Step 3 — apply the affine formula.** `g * x_hat + b` now broadcasts cleanly: every position within channel `c` gets scaled by `gamma[c]` and shifted by `beta[c]`.

**Why it works.** BatchNorm treats each channel as an independent scalar affine line `y = m*x + b`. The reshape is purely about *where* the `C` axis sits; the arithmetic is identical regardless of how many spatial dims there are.

In [ ]:
def bn_affine_1d(x_hat: Tensor, gamma: Tensor, beta: Tensor) -> Tensor:
    g = gamma.view(1, -1, 1)
    b = beta.view(1, -1, 1)
    return g * x_hat + b

t.manual_seed(0)
B, C, L = 2, 3, 4
x_hat = t.randn(B, C, L)
gamma = t.tensor([2.0, 0.5, 1.0])
beta = t.tensor([1.0, -1.0, 0.0])
y = bn_affine_1d(x_hat, gamma, beta)
print(y.shape)
# Channel 2 has gamma=1, beta=0 -> identity
print(t.allclose(y[:, 2, :], x_hat[:, 2, :]))
# Channel 0 scaled by 2 then +1
print(t.allclose(y[:, 0, :], 2.0 * x_hat[:, 0, :] + 1.0))